# Train the land-cover / buildings segmenter, round two (OpenEarthMap + LoveDA)

**Goal:** make the segmenter find buildings it currently misses. Round one (`kaggle_finetune_landcover_openearthmap.ipynb`, OpenEarthMap only) reached validation mIoU 0.633 and building IoU 0.771, and worked well on many real scenes -- but a user's own campus screenshot exposed its weak spot: an image with many large **grey / blue-grey flat concrete roofs** and visible building facades (an off-nadir look) came out as 4% buildings and ~57 outlines, with most grey-roofed blocks labelled "developed space" (paved ground). Grey concrete roofs and pavement look alike from above, and OpenEarthMap's training tiles have relatively few dense urban blocks of that kind.

**Data added:** [LoveDA](https://github.com/Junjue-Wang/LoveDA) (CC BY-NC-SA 4.0, research use), via the Kaggle copy `damifinocchiaro/loveda`: 2,522 train + 1,669 val 1024x1024 RGB tiles at 0.3 m per pixel from Google Earth over three Chinese cities (Nanjing, Changzhou, Wuhan), urban and rural, hand-labelled. Its urban tiles are full of exactly the imagery that failed: high-rise blocks with leaning facades, large flat grey roofs, campuses and factories.

**Design:**
* **Start from round one** (uploaded as the private dataset `satquery-landcover-r1`), so OpenEarthMap knowledge is kept and this run only has to *add* -- fewer epochs than training from scratch.
* **One label space.** LoveDA's classes map onto OpenEarthMap's: building, road, water stay; barren -> bareland; forest -> tree; agricultural -> agriculture land. LoveDA's **"background" is everything else** (grass, paving, plazas, yards...), which OpenEarthMap splits into rangeland and developed space -- so it is *not* mapped to one class. A background pixel is trained with a **set-valued loss**, `-log(P(rangeland) + P(developed space))`: the model may pick either, but any probability it puts on "building" there is penalised. That keeps the crucial negative signal (pavement is not a building) without pretending to know which of the two it is.
* Batches mix both sources; everything else (random-scale crops, U-Net/ResNet34, fp16, cosine schedule) is round one's recipe.
* **Measured against round one, not assumed:** the notebook first scores the round-one weights on LoveDA's validation set, then reports the same numbers after training, plus OpenEarthMap's full validation table (mIoU, per-class IoU, building-count error) to catch any regression there.

## What was checked before writing this (so the cells below aren't guesses)

* **Layout** (Kaggle API): `Train/Train/{Urban,Rural}/{images_png,masks_png}` (1,156 + 1,366 pairs) and `Val/Val/{Urban,Rural}/...` (677 + 992); the copy's test split has no masks.
* **Label encoding**, checked by rendering overlays on six real tiles and looking at them: `0` no data (ignored), `1` background, `2` building, `3` road, `4` water, `5` barren, `6` forest, `7` agricultural. Buildings (red) follow the roofs of the leaning high-rises and the big flat-roofed blocks; roads, the canal and the running track's surroundings come out as expected.
* Images are 1024x1024 RGB PNG, masks 8-bit single-band.

Run cells top to bottom. Kaggle: enable **Internet** and a **GPU**, and attach the three inputs (OpenEarthMap, LoveDA, the round-one checkpoint).

## 0. GPU compatibility check

**Found live, via an actual failed run on Kaggle**: this session's preinstalled PyTorch build
(2.10.0+cu128) only supports CUDA compute capabilities sm_70 and up -- it silently drops support
for the Pascal-generation **P100** (sm_60), one of the two GPU types Kaggle itself still offers
here (T4x2 or P100, per the note below). Landing on a P100 crashed training ~40 seconds in with
`CUDA error: no kernel image is available for execution on the device` -- a real run, not a
hypothetical. The cell below detects the actual GPU via `nvidia-smi` (no torch import needed yet,
so this runs before torch's own compute-capability list is fixed for the process) and reinstalls
a CUDA 11.8 build if the assigned GPU isn't in the preinstalled build's supported list -- CUDA 11.8
wheels cover Pascal through Hopper, so this works regardless of which GPU Kaggle happens to assign.

In [ ]:
import subprocess, sys

try:
    cc_raw = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
    ).strip().splitlines()[0]
    major, minor = cc_raw.split(".")
    needed_sm = f"sm_{major}{minor}"
except Exception as e:
    needed_sm = None
    print(f"Could not query GPU compute capability via nvidia-smi ({e}) -- skipping the compatibility check.")

if needed_sm:
    # Check the INSTALLED build's supported architectures in a SEPARATE PROCESS, not an in-process
    # `import torch` -- Python caches imports in sys.modules, so even an aliased/deleted in-process
    # import here would make a LATER `import torch` in the next cell silently return the stale
    # cached module instead of a fresh one. Confirmed live: an earlier version of this cell did
    # `import torch as _torch_probe`, and the kernel died ~80s after reinstalling -- the old torch
    # stayed resident in this process while its .so files got replaced out from under it on disk.
    check = subprocess.run(
        [sys.executable, "-c",
         "import torch; print(' '.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"],
        capture_output=True, text=True,
    )
    supported = check.stdout.split()
    if needed_sm not in supported:
        print(f"GPU needs {needed_sm}, not in the preinstalled torch build's supported list "
              f"{supported} -- reinstalling a CUDA 11.8 build (covers Pascal through Hopper)...")
        # Uninstall the whole torch/torchvision/torchaudio trio first, then install all three
        # together from the SAME cu118 index in one resolution -- reinstalling `torch` alone
        # left mismatched torchvision/nccl versions behind, which crashed two live runs with
        # unrelated-looking errors (undefined symbol ncclCommShrink; aten.OpaqueObject not
        # registered) that were actually both this same root cause from a different angle.
        subprocess.run(["pip", "uninstall", "-y", "-q", "torch", "torchvision", "torchaudio"], check=True)
        subprocess.run(
            ["pip", "install", "-q", "torch", "torchvision", "torchaudio",
             "--index-url", "https://download.pytorch.org/whl/cu118"],
            check=True,
        )
        print("Reinstalled. The check above ran in a subprocess, so THIS process has never "
              "imported torch itself -- the next cell's `import torch` will be a genuinely fresh "
              "import, not a cached one.")
    else:
        print(f"GPU compute capability {needed_sm} already supported by the preinstalled build.")

In [ ]:
import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(" -", torch.cuda.get_device_name(i))

## 1. Setup

In [ ]:
!pip install -q segmentation-models-pytorch

In [ ]:
import os, sys, json, math, time, random, glob, collections, shutil
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from scipy import ndimage

# Local smoke-test hooks -- never set on Kaggle. LC_DATA_ROOT: a flat folder with images/*.tif + label/*.tif (OpenEarthMap
# pairs); LD_DATA_ROOT: a flat folder with img/*.png + msk/*.png (LoveDA pairs, "_U" in the name = urban); LC_INIT: a
# round-one checkpoint. Together they let the whole notebook run end to end on a laptop before any GPU time is spent.
SMOKE_TEST = bool(os.environ.get("LC_SMOKE_TEST"))
DATA_ROOT = os.environ.get("LC_DATA_ROOT")
LD_ROOT_SMOKE = os.environ.get("LD_DATA_ROOT")
INIT_OVERRIDE = os.environ.get("LC_INIT")

SEED = 0
CLASSES = ["bareland", "rangeland", "developed space", "road", "tree", "water", "agriculture land", "building"]  # OpenEarthMap labels 1..8
NUM_CLASSES, IGNORE = len(CLASSES), 255
BUILDING, RANGELAND, DEVELOPED = CLASSES.index("building"), CLASSES.index("rangeland"), CLASSES.index("developed space")
UNION_BG = 250  # LoveDA "background" -> the pixel is rangeland OR developed space (see the set-valued loss)
# LoveDA raw label -> unified target: 0 no data, 1 background, 2 building, 3 road, 4 water, 5 barren, 6 forest, 7 agricultural
LOVEDA_TO_UNIFIED = np.full(256, IGNORE, dtype=np.int64)
LOVEDA_TO_UNIFIED[[1, 2, 3, 4, 5, 6, 7]] = [UNION_BG, BUILDING, CLASSES.index("road"), CLASSES.index("water"),
                                             CLASSES.index("bareland"), CLASSES.index("tree"), CLASSES.index("agriculture land")]
# the single-class LoveDA labels scored in the LoveDA evaluation: our class index -> LoveDA raw label
LOVEDA_SCORED = {"building": (BUILDING, 2), "road": (CLASSES.index("road"), 3), "water": (CLASSES.index("water"), 4),
                 "tree": (CLASSES.index("tree"), 6), "bareland": (CLASSES.index("bareland"), 5), "agriculture land": (CLASSES.index("agriculture land"), 7)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
CROP = 128 if SMOKE_TEST else 512
CROPS_PER_IMAGE = 1 if SMOKE_TEST else 2
TARGET_BATCH = 2 if SMOKE_TEST else 16
WORKERS = 0 if SMOKE_TEST else 4
MIN_BUILDING_PIXELS = 30

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| smoke test:", SMOKE_TEST)

## 2. Find the data, and the round-one weights to start from

Both dataset roots are found by looking for their folder structure, so a different mount path does not break the notebook. Every label is audited for unexpected values (a wrong encoding would otherwise train silently). If the round-one checkpoint is attached, training starts from it (fewer epochs, lower learning rate); otherwise it starts from ImageNet and trains longer.

In [ ]:
def find_oem_root():
    if DATA_ROOT:
        return DATA_ROOT
    for images_dir in glob.glob("/kaggle/input/**/images", recursive=True):
        root = os.path.dirname(images_dir)
        if os.path.isdir(os.path.join(root, "label")) and os.path.isdir(os.path.join(images_dir, "train")):
            return root
    raise AssertionError("OpenEarthMap not found under /kaggle/input -- add 'aletbm/global-land-cover-mapping-openearthmap' as an input")

def oem_pairs(root, split):
    pattern = f"{root}/images/*.tif" if DATA_ROOT else f"{root}/images/{split}/*.tif"
    return [("oem", p, p.replace("/images/", "/label/")) for p in sorted(glob.glob(pattern)) if os.path.exists(p.replace("/images/", "/label/"))]

def loveda_pairs(split):
    """split: 'Train' or 'Val'. Smoke mode reads a flat folder instead."""
    if LD_ROOT_SMOKE:
        imgs = sorted(glob.glob(f"{LD_ROOT_SMOKE}/img/*.png"))
        pairs = [("loveda", p, p.replace("/img/", "/msk/")) for p in imgs]
        return pairs[:-2] if split == "Train" else pairs[-2:]
    out = []
    for area in ("Urban", "Rural"):
        for img in sorted(glob.glob(f"/kaggle/input/**/{split}/{split}/{area}/images_png/*.png", recursive=True)):
            out.append(("loveda", img, img.replace("images_png", "masks_png")))
    return out

OEM_ROOT = find_oem_root()
if DATA_ROOT:
    every = oem_pairs(OEM_ROOT, None)
    oem_train, oem_val = every[:-2], every[-2:]
else:
    oem_train, oem_val = oem_pairs(OEM_ROOT, "train"), oem_pairs(OEM_ROOT, "val")
ld_train, ld_val = loveda_pairs("Train"), loveda_pairs("Val")
assert oem_train and oem_val and ld_train and ld_val, "a dataset came back empty"
is_urban = lambda p: "/Urban/" in p or "_U." in p
print(f"OpenEarthMap: train {len(oem_train)} | val {len(oem_val)}   LoveDA: train {len(ld_train)} ({sum(is_urban(p) for _, p, _ in ld_train)} urban) | val {len(ld_val)} ({sum(is_urban(p) for _, p, _ in ld_val)} urban)")

# label audits: OpenEarthMap must be 0..8, LoveDA 0..7
def audit(pairs, allowed, k=40):
    counts = np.zeros(256, dtype=np.int64)
    for _, _, label in random.Random(SEED).sample(pairs, min(k, len(pairs))):
        counts += np.bincount(np.array(Image.open(label)).ravel(), minlength=256)
    present = np.nonzero(counts)[0].tolist()
    assert set(present) <= allowed, f"unexpected label values {present}"
    return {v: round(100 * float(counts[v] / counts.sum()), 1) for v in present}
print("OpenEarthMap label shares (%):", audit(oem_train, set(range(9))))
print("LoveDA label shares (%)      :", audit(ld_train, set(range(8))), "(1 background, 2 building, 3 road, 4 water, 5 barren, 6 forest, 7 agricultural)")

INIT = INIT_OVERRIDE or next(iter(glob.glob("/kaggle/input/**/landcover_unet.pt", recursive=True)), None)
print("starting from:", INIT or "ImageNet weights (no round-one checkpoint attached)")
EPOCHS = 1 if SMOKE_TEST else (12 if INIT else 24)
LR = 1.5e-4 if INIT else 3e-4

## 3. Dataset, targets and loss

Every pixel gets a unified target: 0-7 for a single class, **250 for LoveDA "background"** (rangeland or developed space) and 255 for no-data (ignored). Training crops are taken at a random scale (a 256-1000 px window resized to 512, log-uniform), flipped and rotated, with a mild colour jitter only (roof colour is a reported output).

The loss is cross-entropy where a single-class pixel uses the usual `-log P(class)` and a background pixel uses `-log(P(rangeland) + P(developed space))`, plus 0.5 x Dice over the single-class pixels.

In [ ]:
def load_pair(source, image_path, label_path):
    image = np.array(Image.open(image_path).convert("RGB"))
    raw = np.array(Image.open(label_path))
    if raw.ndim == 3:
        raw = raw[..., 0]
    if source == "oem":
        label = np.where(raw == 0, IGNORE, raw.astype(np.int64) - 1)  # 1..8 -> 0..7, unlabelled -> 255
    else:
        label = LOVEDA_TO_UNIFIED[raw]
    return image, label

def to_tensor(image):
    x = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
    return (x - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]

class TrainCrops(Dataset):
    def __init__(self, entries, crops):
        self.entries, self.crops = entries, crops
    def __len__(self):
        return len(self.entries) * self.crops
    def __getitem__(self, i):
        image, label = load_pair(*self.entries[i % len(self.entries)])
        h, w = label.shape
        side = int(math.exp(random.uniform(math.log(min(256, min(h, w))), math.log(min(h, w)))))
        y, x = random.randint(0, h - side), random.randint(0, w - side)
        image, label = image[y:y + side, x:x + side], label[y:y + side, x:x + side]
        img = torch.from_numpy(image).permute(2, 0, 1).float().unsqueeze(0)
        lab = torch.from_numpy(label).float()[None, None]
        img = F.interpolate(img, size=(CROP, CROP), mode="bilinear", align_corners=False, antialias=side > CROP)[0]
        lab = F.interpolate(lab, size=(CROP, CROP), mode="nearest")[0, 0].long()  # nearest keeps 250 / 255 intact
        if random.random() < 0.5: img, lab = img.flip(-1), lab.flip(-1)
        if random.random() < 0.5: img, lab = img.flip(-2), lab.flip(-2)
        k = random.randint(0, 3)
        if k: img, lab = torch.rot90(img, k, (-2, -1)), torch.rot90(lab, k, (-2, -1))
        img = img / 255.0
        img = (img - img.mean()) * random.uniform(0.9, 1.1) + img.mean() * random.uniform(0.9, 1.1)
        gray = img.mean(0, keepdim=True)
        img = (gray + (img - gray) * random.uniform(0.9, 1.1)).clamp(0, 1)
        img = (img - torch.tensor(MEAN)[:, None, None]) / torch.tensor(STD)[:, None, None]
        return img, lab

def make_loader(dataset, batch, shuffle):
    return DataLoader(dataset, batch_size=batch, shuffle=shuffle, drop_last=shuffle, num_workers=WORKERS,
                      pin_memory=(DEVICE == "cuda"), persistent_workers=(WORKERS > 0))

def build_model():
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", classes=NUM_CLASSES, activation=None)

dice_loss = smp.losses.DiceLoss(mode="multiclass", ignore_index=IGNORE)
def loss_fn(logits, target):
    logp = F.log_softmax(logits.float(), dim=1)
    valid = target != IGNORE
    union = target == UNION_BG
    single = valid & ~union
    nll = -logp.gather(1, target.clamp(max=NUM_CLASSES - 1).unsqueeze(1)).squeeze(1)
    nll = torch.where(union, -torch.logsumexp(logp[:, [RANGELAND, DEVELOPED]], dim=1), nll)
    ce = (nll * valid).sum() / valid.sum().clamp(min=1)
    return ce + 0.5 * dice_loss(logits, torch.where(single, target, torch.full_like(target, IGNORE)))

## 4. Metrics

**OpenEarthMap** is scored exactly as in round one (per-class IoU from a confusion matrix, area-fraction error, building-count error, half scale) so the two rounds are directly comparable. **LoveDA** is scored per single class as a binary IoU (predicted class vs LoveDA's label; a *building* prediction on a "background" pixel counts as a false positive), plus building precision and recall -- the numbers that say whether missed grey roofs are found without inventing buildings -- plus how often "background" pixels are called rangeland or developed space, and building IoU on the urban tiles alone.

In [ ]:
def confusion(pred, target):
    valid = target != IGNORE
    idx = target[valid] * NUM_CLASSES + pred[valid]
    return torch.bincount(idx, minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES).cpu().numpy()

def iou_from_confusion(cm):
    tp = np.diag(cm).astype(float)
    denom = cm.sum(0) + cm.sum(1) - np.diag(cm)
    return np.where(denom > 0, tp / np.maximum(denom, 1), np.nan)

def count_components(mask, min_pixels=MIN_BUILDING_PIXELS):
    labelled, n = ndimage.label(mask, structure=np.ones((3, 3)))
    if n == 0:
        return 0
    sizes = np.bincount(labelled.ravel())[1:]
    return int((sizes >= min_pixels).sum())

@torch.no_grad()
def predict_full(model, image):
    x = to_tensor(image)[None].to(DEVICE)
    h, w = x.shape[-2:]
    x = F.pad(x, (0, (-w) % 32, 0, (-h) % 32), mode="reflect")
    with torch.autocast("cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
        logits = model(x)
    return logits.float()[..., :h, :w].argmax(1)[0]

def evaluate_oem(model, pairs, scale=1.0, with_counts=False):
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    frac_err = np.zeros(NUM_CLASSES); n_tiles = 0
    pred_counts, true_counts = [], []
    for source, image_path, label_path in pairs:
        image, label = load_pair(source, image_path, label_path)
        if scale != 1.0:
            h, w = label.shape
            size = (int(w * scale), int(h * scale))
            image = np.array(Image.fromarray(image).resize(size, Image.LANCZOS))
            label = np.array(Image.fromarray(label.astype(np.uint8)).resize(size, Image.NEAREST)).astype(np.int64)
        pred = predict_full(model, image)
        target = torch.from_numpy(label).to(DEVICE)
        cm += confusion(pred, target)
        valid = target != IGNORE
        for c in range(NUM_CLASSES):
            frac_err[c] += abs(float(((pred == c) & valid).sum() - ((target == c) & valid).sum())) / max(int(valid.sum()), 1)
        n_tiles += 1
        if with_counts:
            pred_counts.append(count_components((pred == BUILDING).cpu().numpy()))
            true_counts.append(count_components((target == BUILDING).cpu().numpy()))
    iou = iou_from_confusion(cm)
    out = {"mIoU": float(np.nanmean(iou)), "iou": {CLASSES[c]: (None if np.isnan(iou[c]) else float(iou[c])) for c in range(NUM_CLASSES)},
           "pixel_accuracy": float(np.diag(cm).sum() / max(cm.sum(), 1)),
           "area_fraction_error_pct": {CLASSES[c]: 100 * float(frac_err[c] / max(n_tiles, 1)) for c in range(NUM_CLASSES)}}
    if with_counts and true_counts:
        p, t = np.array(pred_counts, float), np.array(true_counts, float)
        out["building_count"] = {"mean_true": float(t.mean()), "mean_pred": float(p.mean()), "mean_abs_error": float(np.abs(p - t).mean()),
                                 "median_relative_error": float(np.median(np.abs(p - t) / np.maximum(t, 1))),
                                 "correlation": float(np.corrcoef(p, t)[0, 1]) if len(t) > 2 and t.std() > 0 and p.std() > 0 else None}
    return out

def evaluate_loveda(model, pairs):
    model.eval()
    acc = {k: collections.Counter() for k in ("all", "urban")}
    bg_hit = bg_total = 0
    for source, image_path, label_path in pairs:
        image = np.array(Image.open(image_path).convert("RGB"))
        raw = np.array(Image.open(label_path))
        if raw.ndim == 3:
            raw = raw[..., 0]
        pred = predict_full(model, image).cpu().numpy()
        valid = raw != 0
        groups = ["all"] + (["urban"] if is_urban(image_path) else [])
        for name, (ours, theirs) in LOVEDA_SCORED.items():
            p, t = (pred == ours) & valid, (raw == theirs) & valid
            for g in groups:
                acc[g][name, "tp"] += int((p & t).sum()); acc[g][name, "fp"] += int((p & ~t).sum()); acc[g][name, "fn"] += int((~p & t).sum())
        background = raw == 1
        bg_total += int(background.sum()); bg_hit += int((background & ((pred == RANGELAND) | (pred == DEVELOPED))).sum())
    def scores(g):
        out = {}
        for name in LOVEDA_SCORED:
            tp, fp, fn = acc[g][name, "tp"], acc[g][name, "fp"], acc[g][name, "fn"]
            out[name] = {"iou": tp / max(tp + fp + fn, 1), "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1)}
        out["macro_iou"] = float(np.mean([out[n]["iou"] for n in LOVEDA_SCORED]))
        return out
    result = scores("all")
    result["urban"] = scores("urban") if acc["urban"] else None
    result["background_called_rangeland_or_developed"] = bg_hit / max(bg_total, 1)
    return result

## 5. Pre-flight: the whole train / save / reload path, before the long run

Runs the *same* functions the real run uses -- model build, two fp16-autocast optimiser steps with gradient scaling on a batch that exercises the set-valued loss, whole-tile prediction, save and reload -- on synthetic data first, then **finds the largest batch that fits** (15% of GPU memory kept free).

In [ ]:
def synthetic_targets(n, size):
    y = torch.randint(0, NUM_CLASSES, (n, size, size), device=DEVICE)
    y[:, : size // 4] = UNION_BG  # LoveDA-style background pixels
    y[:, :2] = IGNORE
    return y

def preflight():
    t0 = time.time()
    use_amp = DEVICE == "cuda"
    m = build_model().to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    x = torch.randn(2, 3, CROP, CROP, device=DEVICE)
    y = synthetic_targets(2, CROP)
    m.train()
    for _ in range(2):
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(m(x), y)
        assert torch.isfinite(loss), f"non-finite loss ({float(loss)}) -- fp16 is overflowing"
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    pred = predict_full(m, np.random.randint(0, 255, (300, 260, 3), dtype=np.uint8))
    assert tuple(pred.shape) == (300, 260) and int(pred.max()) < NUM_CLASSES
    path = "/tmp/preflight_lc2.pt"
    torch.save({"model_state_dict": m.state_dict()}, path)
    m2 = build_model(); m2.load_state_dict(torch.load(path, map_location="cpu")["model_state_dict"]); os.remove(path)
    del m, m2, opt
    if DEVICE == "cuda": torch.cuda.empty_cache()
    print(f"pre-flight OK in {time.time() - t0:.0f}s: model build, fp16 steps with the set-valued loss, tile prediction, save/reload")

preflight()

def pick_batch(target):
    if DEVICE != "cuda":
        return target
    m = build_model().to(DEVICE); m.train()
    scaler = torch.amp.GradScaler("cuda")
    total = torch.cuda.get_device_properties(0).total_memory
    chosen = None
    for cand in [c for c in (24, 16, 12, 8, 4, 2, 1) if c <= max(target, 8)]:
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        try:
            x = torch.randn(cand, 3, CROP, CROP, device=DEVICE)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(m(x), synthetic_targets(cand, CROP))
            scaler.scale(loss).backward()
            peak = torch.cuda.max_memory_allocated()
        except torch.cuda.OutOfMemoryError:
            peak = total
        for p in m.parameters(): p.grad = None
        x = loss = None
        print(f"  batch {cand:2d}: peak {peak / 1e9:5.1f} GB of {total / 1e9:.1f} GB -> {'fits' if peak < 0.85 * total else 'too big'}")
        if peak < 0.85 * total:
            chosen = cand; break
    assert chosen, "not even batch 1 fits"
    del m, scaler
    torch.cuda.empty_cache()
    return chosen

BATCH = pick_batch(TARGET_BATCH)
print("batch size:", BATCH)

## 6. Baseline: the round-one weights on LoveDA

The round-one checkpoint scored on LoveDA's validation tiles (up to 400 of them, to keep it short), before any training here -- so what this round adds is a measured difference, not an impression. It also confirms the checkpoint loads into this architecture.

In [ ]:
model = build_model().to(DEVICE)
baseline = None
if INIT:
    ck = torch.load(INIT, map_location="cpu", weights_only=False)
    assert list(ck["classes"]) == CLASSES, "the starting checkpoint has different classes"
    model.load_state_dict(ck["model_state_dict"])
    ld_subset_baseline = random.Random(SEED).sample(ld_val, min(400, len(ld_val)))
    baseline = evaluate_loveda(model, ld_subset_baseline)
    b = baseline["building"]
    u = baseline["urban"]["building"]["iou"] if baseline["urban"] else float("nan")
    print(f"round one on LoveDA val ({len(ld_subset_baseline)} tiles): macro IoU {baseline['macro_iou']:.3f} | building IoU {b['iou']:.3f} precision {b['precision']:.3f} recall {b['recall']:.3f} | urban building IoU {u:.3f}")
    print(f"  'background' called rangeland or developed space: {baseline['background_called_rangeland_or_developed']:.1%}")
else:
    print("no starting checkpoint -- training from ImageNet weights")

## 7. Train

AdamW (encoder at one third of the learning rate), warm-up then cosine decay, fp16 autocast. Each epoch is scored on a fixed subset of each dataset's validation tiles (128 OpenEarthMap, 200 LoveDA) and the weights with the best **mean of OpenEarthMap mIoU and LoveDA macro IoU** are kept, so neither dataset can be traded away for the other. A non-finite loss skips the step; more than 5% of them aborts.

In [ ]:
train_entries = oem_train + ld_train
if SMOKE_TEST:
    train_entries = train_entries[:4] + ld_train[:2]
train_loader = make_loader(TrainCrops(train_entries, CROPS_PER_IMAGE), BATCH, shuffle=True)
oem_val_subset = random.Random(SEED).sample(oem_val, min(128, len(oem_val)))
ld_val_subset = random.Random(SEED + 1).sample(ld_val, min(200, len(ld_val)))
encoder_params = list(model.encoder.parameters())
encoder_ids = {id(p) for p in encoder_params}
other_params = [p for p in model.parameters() if id(p) not in encoder_ids]
optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": LR / 3}, {"params": other_params, "lr": LR}], weight_decay=1e-4)
steps_per_epoch = len(train_loader)
steps_total = EPOCHS * steps_per_epoch
warmup = max(1, int(0.03 * steps_total))
def lr_factor(step):
    if step < warmup:
        return (step + 1) / warmup
    progress = (step - warmup) / max(1, steps_total - warmup)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

CKPT_DIR = "/tmp/lc2_ckpt" if SMOKE_TEST else "/kaggle/working/lc2_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
best_sel, best_state, skipped, step, history = -1.0, None, 0, 0, []
t_start = time.time()
print(f"{len(train_entries)} training tiles ({len(oem_train)} OpenEarthMap + {len(ld_train)} LoveDA); {steps_total} optimiser steps ({steps_per_epoch} per epoch), batch {BATCH}, lr {LR}")
for epoch in range(EPOCHS):
    model.train()
    running, n, t_epoch = 0.0, 0, time.time()
    for images, labels in train_loader:
        images, labels = images.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss = loss_fn(model(images), labels)
        if not torch.isfinite(loss):
            skipped += 1
            optimizer.zero_grad(set_to_none=True)
            assert skipped <= max(3, 0.05 * (step + 1)), "too many non-finite losses -- fp16 is overflowing"
            continue
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item(); n += 1; step += 1
        if step % 100 == 0:
            print(f"  step {step}/{steps_total}  loss {running / max(n, 1):.4f}  ({time.time() - t_start:.0f}s)", flush=True)
    oem_r, ld_r = evaluate_oem(model, oem_val_subset), evaluate_loveda(model, ld_val_subset)
    sel = 0.5 * oem_r["mIoU"] + 0.5 * ld_r["macro_iou"]
    history.append({"epoch": epoch + 1, "train_loss": running / max(n, 1), "oem_mIoU": oem_r["mIoU"], "loveda_macro_iou": ld_r["macro_iou"], "loveda_building_iou": ld_r["building"]["iou"], "selection": sel})
    marker = ""
    if sel > best_sel:
        best_sel, best_state, marker = sel, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "  <- best"
        torch.save({"model_state_dict": best_state}, os.path.join(CKPT_DIR, "best_state.pt"))
    print(f"epoch {epoch + 1}/{EPOCHS}  train loss {running / max(n, 1):.4f}  OEM mIoU {oem_r['mIoU']:.4f}  LoveDA macro IoU {ld_r['macro_iou']:.4f} (building {ld_r['building']['iou']:.3f})  select {sel:.4f}  ({time.time() - t_epoch:.0f}s){marker}", flush=True)
print(f"\ntraining took {(time.time() - t_start) / 60:.1f} min; best selection score {best_sel:.4f}; skipped {skipped} non-finite steps")
model.load_state_dict(best_state)

## 8. Final evaluation

The complete OpenEarthMap validation set exactly as in round one (round one: mIoU 0.633, building IoU 0.771, pixel accuracy 0.777, half-scale mIoU 0.591, building count true 75.5 / predicted 74.3, correlation 0.883), and the complete LoveDA validation set, before and after.

In [ ]:
oem_final = evaluate_oem(model, oem_val, scale=1.0, with_counts=True)
oem_half = evaluate_oem(model, oem_val, scale=0.5)
print(f"\nOPENEARTHMAP VALIDATION ({len(oem_val)} tiles)  mIoU {oem_final['mIoU']:.4f}  pixel accuracy {oem_final['pixel_accuracy']:.4f}   |   half scale: mIoU {oem_half['mIoU']:.4f}   (round one: 0.6331 / 0.7767 / 0.5906)")
print(f"  {'class':18s} {'IoU':>7s} {'IoU @0.5x':>10s} {'area-fraction error (pct points)':>34s}")
for c in CLASSES:
    a, b = oem_final["iou"][c], oem_half["iou"][c]
    print(f"  {c:18s} {('n/a' if a is None else f'{a:.3f}'):>7s} {('n/a' if b is None else f'{b:.3f}'):>10s} {oem_final['area_fraction_error_pct'][c]:>34.2f}")
bc = oem_final.get("building_count")
if bc:
    print(f"  building count per tile: true {bc['mean_true']:.1f}, predicted {bc['mean_pred']:.1f}, MAE {bc['mean_abs_error']:.1f}, median relative error {bc['median_relative_error']:.0%}, correlation {bc['correlation'] if bc['correlation'] is None else round(bc['correlation'], 3)}")

ld_final = evaluate_loveda(model, ld_val)
print(f"\nLOVEDA VALIDATION ({len(ld_val)} tiles)  macro IoU {ld_final['macro_iou']:.4f}   'background' called rangeland or developed space: {ld_final['background_called_rangeland_or_developed']:.1%}")
print(f"  {'class':18s} {'IoU':>7s} {'precision':>10s} {'recall':>8s} {'urban IoU':>10s}")
for name in LOVEDA_SCORED:
    r = ld_final[name]; u = ld_final["urban"][name]["iou"] if ld_final["urban"] else float("nan")
    print(f"  {name:18s} {r['iou']:7.3f} {r['precision']:10.3f} {r['recall']:8.3f} {u:10.3f}")
if baseline:
    b0, b1 = baseline["building"], ld_final["building"]
    u0 = baseline["urban"]["building"]["iou"] if baseline["urban"] else float("nan")
    u1 = ld_final["urban"]["building"]["iou"] if ld_final["urban"] else float("nan")
    print(f"\nBUILDINGS on LoveDA, round one -> now: IoU {b0['iou']:.3f} -> {b1['iou']:.3f} | precision {b0['precision']:.3f} -> {b1['precision']:.3f} | recall {b0['recall']:.3f} -> {b1['recall']:.3f} | urban IoU {u0:.3f} -> {u1:.3f}")
    print(f"(round one was scored on {len(ld_subset_baseline)} of the {len(ld_val)} validation tiles)")

## 9. Export

The same checkpoint dict as round one (so `models/landcover/landcover_tool.py` needs no change), with both rounds' numbers in `metrics`, and a JSON copy.

In [ ]:
OUT_DIR = "/tmp/lc2_out" if SMOKE_TEST else "/kaggle/working/lc2_out"
os.makedirs(OUT_DIR, exist_ok=True)
metrics = {"oem_val_tiles": len(oem_val), "loveda_val_tiles": len(ld_val), "oem_native": oem_final, "oem_half_scale": oem_half, "loveda": ld_final,
           "loveda_round_one_baseline": baseline, "history": history, "epochs": EPOCHS, "started_from": INIT,
           "train_tiles": {"openearthmap": len(oem_train), "loveda": len(ld_train)}}
torch.save({
    "model_state_dict": best_state, "encoder_name": "resnet34", "classes": CLASSES, "num_classes": NUM_CLASSES,
    "train_crop": CROP, "mean": MEAN, "std": STD, "metrics": metrics,
    "label_note": "OpenEarthMap classes in this order; LoveDA background was trained as 'rangeland or developed space'",
}, os.path.join(OUT_DIR, "landcover_unet.pt"))
json.dump(metrics, open(os.path.join(OUT_DIR, "landcover_metrics.json"), "w"), indent=1)
print("exported:", sorted(os.listdir(OUT_DIR)), f"({os.path.getsize(os.path.join(OUT_DIR, 'landcover_unet.pt')) / 1e6:.0f} MB)")

ck = torch.load(os.path.join(OUT_DIR, "landcover_unet.pt"), map_location="cpu", weights_only=False)
check = build_model(); check.load_state_dict(ck["model_state_dict"]); check.to(DEVICE)
image, _ = load_pair(*oem_val[0])
pred = predict_full(check, image)
print("artifact round-trip OK:", tuple(pred.shape), "predicted classes:", sorted(set(pred.cpu().numpy().ravel().tolist())))